# Homework 04 — Data Acquisition & Ingestion

**Anna Cui** · FRE 5040

Adapted from `stage04_data-acquisition-and-ingestion_homework-starter.ipynb`.
Two ingestion paths — a market-data **API** pull and a **scraped** reference
table — both validated and saved to `data/raw/` with reproducible filenames.

> Ethics and legality: obeys site Terms, `robots.txt` and rate limits. Sends a
> descriptive User-Agent, makes single requests, never loops.

In [1]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()
CHECKS = [
    (".env",         "NEEDED", "YOU create this: copy .env.example to .env"),
    (".env.example", "NEEDED", "the committed template"),
]
print(f"Looking in: {ROOT}\n")
for rel, kind, note in CHECKS:
    print(f"  [{'OK ' if (ROOT / rel).exists() else 'MISS'}]  {kind:<8}  {rel:<16}  {note}")

Looking in: /Users/annacui/NYU/Bootcamp/Bootcamp 4/bootcamp_Anna_Cui/homework/homework04

  [OK ]  NEEDED    .env              YOU create this: copy .env.example to .env
  [OK ]  NEEDED    .env.example      the committed template


In [2]:
import os, pathlib, datetime as dt, subprocess
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

RAW = pathlib.Path('data/raw'); RAW.mkdir(parents=True, exist_ok=True)
load_dotenv(); print('ALPHAVANTAGE_API_KEY loaded?', bool(os.getenv('ALPHAVANTAGE_API_KEY')))

ALPHAVANTAGE_API_KEY loaded? True


## Helpers — the starter's, with three documented changes

**1. `ts()` drops seconds.** The starter used `%Y%m%d-%H%M%S`; the sheet's
documented deliverable is `..._<YYYYMMDD-HHMM>.csv`. Matching the spec costs a
collision if this is re-run inside the same minute — accepted, and recorded
here rather than left as a surprise.

**2. `save_csv` joins values, not `key-value` pairs**, so output reads
`api_yfinance_VTI_20260828-0830.csv` as the sheet shows.

**3. `validate` gains dtype coercion and a duplicate-row count.** The starter
checked columns, shape and NA total — enough to catch a missing column, not
enough to catch a price column that arrived as text, which is the failure that
survives into modeling.

In [3]:
def ts():
    return dt.datetime.now().strftime('%Y%m%d-%H%M')


def save_csv(df: pd.DataFrame, prefix: str, **meta):
    mid = '_'.join(str(v) for v in meta.values())
    path = RAW / f"{prefix}_{mid}_{ts()}.csv"
    df.to_csv(path, index=False)
    print('Saved', path, f'({len(df)} rows)')
    return path


def validate(df: pd.DataFrame, required, dtypes_map=None):
    missing = [col for col in required if col not in df.columns]
    out = {'missing': missing,
           'shape': df.shape,
           'na_total': int(df.isna().sum().sum()),
           'duplicate_rows': int(df.duplicated().sum())}
    for col, kind in (dtypes_map or {}).items():
        if col in df.columns:
            try:
                pd.to_datetime(df[col]) if kind == 'datetime' else pd.to_numeric(df[col])
                out[f'dtype_{col}'] = 'ok'
            except Exception as exc:
                out[f'dtype_{col}'] = f'FAILED: {exc}'
    return out

## Part 1 — API Pull

### Two things in the starter that do not run, and why

**`TIME_SERIES_DAILY_ADJUSTED` is a premium endpoint.** Adjusted close moved
behind Alpha Vantage's paywall. The free tier answers `200 OK` with prose
instead of data, so `raise_for_status()` passes and the starter's
`[k for k in js if 'Time Series' in k][0]` raises `IndexError` on an empty list.
Fixed by requesting `TIME_SERIES_DAILY`, taking raw `close`, and **checking for
the series rather than for the absence of an error**.

**`yf.download(...)` defaults to `auto_adjust=True`**, which returns no
`Adj Close` column at all, so the starter's `[['Date','Adj Close']]` raises
`KeyError`. Fixed with `auto_adjust=False` plus a column-shape check, since
yfinance returns a MultiIndex under some version and argument combinations.

Both are one lesson: **code against the shape of what comes back.**

In [4]:
SYMBOL = 'VTI'          # the US equity sleeve of the 60/30/10 model portfolio

ALPHA_KEY = os.getenv('ALPHAVANTAGE_API_KEY')
# A placeholder key is not a usable key - treat it as absent rather than
# discovering that four requests later.
USE_ALPHA = bool(ALPHA_KEY) and ALPHA_KEY != 'dummy_key_123'
print('Using Alpha Vantage:', USE_ALPHA, '| yfinance fallback:', not USE_ALPHA)

if USE_ALPHA:
    url = 'https://www.alphavantage.co/query'
    params = {'function': 'TIME_SERIES_DAILY', 'symbol': SYMBOL,
              'outputsize': 'compact', 'apikey': ALPHA_KEY, 'datatype': 'json'}
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()                     # passes even when the daily cap is hit
    js = r.json()
    keys = [k for k in js if 'Time Series' in k]
    if not keys:
        print('Alpha Vantage returned no series:', str(list(js.values())[0])[:150])
        USE_ALPHA = False
    else:
        df_api = (pd.DataFrame(js[keys[0]]).T.rename_axis('date').reset_index()
                    [['date', '4. close']].rename(columns={'4. close': 'close'}))
        SOURCE = 'alphavantage'

if not USE_ALPHA:
    import yfinance as yf
    raw = yf.download(SYMBOL, period='6mo', interval='1d',
                      auto_adjust=False, progress=False)
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    df_api = raw.reset_index()[['Date', 'Close']]
    df_api.columns = ['date', 'close']
    SOURCE = 'yfinance'

df_api['date'] = pd.to_datetime(df_api['date'])
df_api['close'] = pd.to_numeric(df_api['close'])
df_api = df_api.sort_values('date').reset_index(drop=True)
df_api.head()

Using Alpha Vantage: False | yfinance fallback: True


,date,close
0,2026-03-02,339.119995
1,2026-03-03,335.730011
2,2026-03-04,338.190002
3,2026-03-05,336.000000
4,2026-03-06,331.410004


In [5]:
v_api = validate(df_api, required=['date', 'close'],
                 dtypes_map={'date': 'datetime', 'close': 'numeric'})
v_api

{'missing': [],
 'shape': (125, 2),
 'na_total': 0,
 'duplicate_rows': 0,
 'dtype_date': 'ok',
 'dtype_close': 'ok'}

In [6]:
# Sanity rules beyond schema: prices positive, dates ordered and unique.
assert (df_api['close'] > 0).all(),            'non-positive close price'
assert df_api['date'].is_monotonic_increasing, 'dates are not sorted'
assert not df_api['date'].duplicated().any(),  'duplicate trading dates'
print(f"{len(df_api)} rows, {df_api['date'].min().date()} to {df_api['date'].max().date()} - passed")

125 rows, 2026-03-02 to 2026-08-27 - passed


In [7]:
api_path = save_csv(df_api, 'api', source=SOURCE, ticker=SYMBOL)

Saved data/raw/api_yfinance_VTI_20260828-0852.csv (125 rows)


## Part 2 — Scrape a permitted public table

The starter's `SCRAPE_URL` is the placeholder `https://example.com/markets-table`,
which does not exist. Replaced with Wikipedia's S&P 500 constituents list: a
public, permitted page with a stable table id, and relevant here because VTI
tracks the total US market, so this is the closest public description of what
the equity sleeve holds.

**Resilient selectors:** semantic id first, generic class second, inline HTML
third. Any selector will eventually break; `scrape_ok` records which path ran so
a fallback cannot masquerade as a successful scrape.

In [8]:
SCRAPE_URL = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'FRE5040-coursework/1.0 (student project; ac13155@nyu.edu)'}


def rows_from(node):
    return [[cell.get_text(strip=True) for cell in tr.find_all(['th', 'td'])]
            for tr in node.find_all('tr') if tr.find_all(['th', 'td'])]


try:
    resp = requests.get(SCRAPE_URL, headers=headers, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    table = soup.find('table', id='constituents') or soup.find('table', class_='wikitable')
    if table is None:
        raise RuntimeError('no constituents table - page structure changed')
    header, *data = rows_from(table)
    df_scrape = pd.DataFrame(data, columns=header)
    scrape_ok = True
except Exception as e:
    print('Scrape failed, using inline demo table:', e)
    html = '<table><tr><th>Symbol</th><th>Security</th></tr>' \
           '<tr><td>AAA</td><td>Placeholder Corp</td></tr></table>'
    header, *data = rows_from(BeautifulSoup(html, 'html.parser'))
    df_scrape = pd.DataFrame(data, columns=header)
    scrape_ok = False

print('scraped the live page:', scrape_ok, '| shape:', df_scrape.shape)
df_scrape.head()

scraped the live page: True | shape: (503, 8)


,Symbol,Security,GICSSector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,0000066740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,0000091142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,0000001800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,0001551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,0001467373,1989


In [9]:
# Coerce defensively. errors='coerce' turns unparseable values into NaN/NaT
# instead of raising, and na_total below then reports how many failed - a silent
# float-parsing failure is the pitfall the reading names explicitly.
df_scrape.columns = [str(col).strip() for col in df_scrape.columns]

for col in df_scrape.columns:
    if 'date' in col.lower():
        df_scrape[col] = pd.to_datetime(df_scrape[col], errors='coerce')
    elif col.upper() == 'CIK':
        df_scrape[col] = pd.to_numeric(df_scrape[col], errors='coerce')

v_scrape = validate(df_scrape, required=list(df_scrape.columns)[:2])
v_scrape

{'missing': [], 'shape': (503, 8), 'na_total': 0, 'duplicate_rows': 0}

In [10]:
scrape_path = save_csv(df_scrape, 'scrape', site='wikipedia', table='sp500')

Saved data/raw/scrape_wikipedia_sp500_20260828-0852.csv (503 rows)


## Documentation

| | API | Scrape |
|---|---|---|
| Source | Alpha Vantage `TIME_SERIES_DAILY`, falling back to yfinance | `en.wikipedia.org/wiki/List_of_S%26P_500_companies` |
| Parameters | `symbol=VTI`, `period=6mo`, `interval=1d`, `auto_adjust=False` | table `id='constituents'`, fallback `class='wikitable'` |
| Auth | `ALPHAVANTAGE_API_KEY` from `.env` | none |
| Field taken | raw `close` — **not** adjusted close, a paid field | all columns as published |

**Validation logic.** Schema (required columns) · types (dates parse, prices
coerce to numeric, reported per column) · completeness (NA total, shape,
duplicate rows) · sanity (prices strictly positive, dates sorted and unique,
asserted rather than eyeballed).

**Secrets.** `.env` holds the key and is excluded by the repository
`.gitignore`; `.env.example` is committed as the template. Verified below,
because "I remembered to gitignore it" is not evidence.

In [11]:
for target in ['.env', '.env.example']:
    result = subprocess.run(['git', 'check-ignore', '-v', target],
                            capture_output=True, text=True)
    print(f"{target:<14} -> {result.stdout.strip() or '(not ignored - correct for the template)'}")

.env           -> .gitignore:5:.env	.env
.env.example   -> (not ignored - correct for the template)


## Assumptions & risks

- **Rate limits.** Alpha Vantage's free tier allows 25 calls/day and signals
  exhaustion with `200 OK` and prose, not an error status. Trusting the status
  code saves an empty file and reports success.
- **Selector fragility.** `id='constituents'` is stable today and Wikipedia
  editors are under no obligation to keep it. Two fallbacks exist — but a
  fallback that quietly yields two placeholder rows is its own hazard, so
  `scrape_ok` records which path ran.
- **Schema drift.** Vendors rename, move and paywall fields; this pull takes raw
  `close` precisely because `adjusted close` moved behind a paywall. Raw close
  means dividends and splits are **not** reflected — acceptable for short-window
  weight drift, wrong for long-horizon return calculations.
- **Silent float parsing.** Currency symbols and thousands separators defeat
  `to_numeric`; `errors='coerce'` surfaces them in the NA count instead of
  hiding them.
- **Not point-in-time.** The constituents list is today's membership with no
  add/remove history. Applying it to older prices would be survivorship bias.
- **Filename collisions.** `ts()` is minute-resolution to match the sheet, so
  two runs inside one minute overwrite. Seconds would be safer; the spec wins,
  and the tradeoff is recorded rather than hidden.
- **Timestamped, not overwritten.** A later run adds a file rather than
  replacing one, so an earlier state stays recoverable.

In [12]:
for path in sorted(RAW.glob('*.csv')):
    print(f"{path.name:<48} {path.stat().st_size:>9,} bytes")

api_yfinance_VTI_20260828-0852.csv                   3,619 bytes
scrape_wikipedia_sp500_20260828-0852.csv            53,609 bytes
